[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/pff-damage.ipynb)

# Phase-Field Fracture on a Random-Fibre 3-Phase Microstructure

A walkthrough of FFTjax's staggered phase-field fracture solver
(`solvers.mechanical.strain_nw_cg.solve_elastic` + `solvers.damage.pff_damage.solve_helmholtz_cg_het`)
on a **randomly-packed circular-fibre composite** with three phases (matrix, interphase, fibre) --
generated with `generation.rve.make_random_composite_rve` (Catalanotti 2016) rather than loaded
from a real segmented scan (see `data/microstructures/README` for that real-patch route, used by
`configs/user/pff_prototype.yaml`).

Material constants and phase-field parameters below are the same ones already used in
`configs/user/pff_prototype.yaml`, the project's reference config for the real-patch dataset -- see
that file for the full production setup (adaptive timestepper, YAML-driven, XDMF output). This
notebook strips that down to a minimal, directly-runnable walkthrough on a synthetic RVE instead,
so it needs no external data file.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    print("Running locally — using the local src/ checkout.")
    import sys
    sys.path.insert(0, "../src")

In [ ]:
import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from mat_models.elastic import (
    LinearElasticIsotropic, TransverseIsotropicFibre,
    assemble_C_field_oriented, lame_from_C_field, strain_energy_amor_split,
)
from operators.green import build_freq_grid, build_green_operator
from solvers.mechanical.strain_nw_cg import solve_elastic
from solvers.damage.pff_damage import degradation, update_history_hybrid, solve_helmholtz_cg_het
from post.fields import field_to_grid, von_mises
from utils.io.xdmf_writer import IncrementalWriter
from post.fields import to_voigt

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate a random-fibre RVE

Instead of loading a real segmented patch, this generates an idealized microstructure with
`generation.rve.make_random_composite_rve` (Catalanotti 2016) -- randomly-packed circular fibres
with a thin interphase ring around each one, same generator driving
`scripts/generation/generate_rve.py`/`configs/generation/rve_random.yaml`.

The generator's own phase labeling is `0=matrix, 1=fibre, 2=interphase`; remapped below to
`0=matrix, 1=interphase, 2=fibre` to match this notebook's materials list (`[matrix, interphase,
fiber]`) and `configs/user/pff_prototype.yaml`'s real-patch convention. Unlike a real patch, every
fibre here shares one global orientation (along Z) rather than a per-voxel field measured from an
actual cross-section.

In [ ]:
from generation.rve import make_random_composite_rve

phi_target = 0.55           # target fibre volume fraction
r_fiber    = 0.0035         # fibre radius [mm]
vox        = 0.0001         # target voxel size [mm]
interphase_thickness = 0.0002   # mm

phase_raw, n, L, phi_fiber_act, centres = make_random_composite_rve(
    phi=phi_target,
    r_fiber=r_fiber,
    dx=vox,
    size_in_r=15,   # domain side ~ 15*r_fiber (Catalanotti 2016 convention)
    nz=1,
    K=15,           # perturbation iterations (K>10 -> fully randomised)
    seed=67,
    interphase_thickness=interphase_thickness,
)

# remap 0=matrix,1=fibre,2=interphase (generator's own labeling) -> 0=matrix,
# 1=interphase,2=fibre (this notebook's convention, see markdown above)
phase_np = np.where(phase_raw == 1, 2, np.where(phase_raw == 2, 1, 0)).astype(np.uint8).reshape(-1)

Nv = int(np.prod(n))
dx = tuple(Li / ni for Li, ni in zip(L, n))

# idealized RVE: every fibre shares one global axis (along Z), unlike a real
# patch's per-voxel orientation field
fibre_dir = np.array([0.0, 0.0, 1.0])
orientation_np = np.tile(fibre_dir[:, None], (1, Nv))   # (3, Nv)

phi = float(((phase_np == 1) | (phase_np == 2)).mean())   # fibre+interphase volume fraction

print("Generated random-fibre RVE (Catalanotti 2016)")
print("grid n :", n)
print("domain L [mm]:", L)
print("phi (fibre+interphase volume fraction):", phi)
print("phase fractions -- matrix:", float((phase_np == 0).mean()),
      " interphase:", float((phase_np == 1).mean()),
      " fibre:", float((phase_np == 2).mean()))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(phase_np.reshape(n)[:, :, 0].T, origin="lower", cmap="plasma",
               extent=[0, n[0]*dx[0], 0, n[1]*dx[1]])
ax.set_title(f"Phase (0=matrix, 1=interphase, 2=fibre), phi={phi:.3f}")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
fig.colorbar(im, ax=ax, ticks=[0, 1, 2])
plt.show()

## Materials and stiffness field

Same constants as `configs/user/pff_prototype.yaml`: an epoxy matrix, an interphase layer (the
config's placeholder modulus -- between the matrix and the fibre's transverse modulus, since
there's no real scan to calibrate it against for this synthetic RVE), and a transversely isotropic
carbon fibre using typical literature constants.

`assemble_C_field_oriented` rotates the fibre's stiffness tensor per voxel using the generated
`orientation` field (uniform along Z here, see the RVE-generation cell above).

In [ ]:
matrix = LinearElasticIsotropic(E=3500.0, nu=0.35, name="epoxy matrix")
interphase = LinearElasticIsotropic(E=12000.0, nu=0.35, name="interphase")
fiber = TransverseIsotropicFibre(
    E_L=230000.0, E_T=15000.0, G_LT=15000.0, nu_LT=0.20, nu_TT=0.30, name="carbon fibre",
)

phase = jnp.array(phase_np)
orientation = jnp.array(orientation_np)
matrix_mask = phase == 0
interphase_mask = phase == 1
damage_zone = matrix_mask | interphase_mask   # fibre never damages

C_field = assemble_C_field_oriented([matrix, interphase, fiber], phase, orientation)
lam_vox, mu_vox = lame_from_C_field(C_field)

print(matrix)
print(interphase)
print(fiber)

## Frequency grid, Green's operator, and AT2 parameters

Reference medium is the (matrix, fibre) Lame-parameter average, matching the other composite
examples. `Gc_field` is heterogeneous: the interphase cracks more easily than the matrix
(`Gc_interphase < Gc_mat`), and the fibre's toughness is set high enough that it doesn't crack
under the strains applied here -- the same values as `configs/pff_prototype.yaml`.

In [ ]:
xi_flat = build_freq_grid(n, L)

lam0 = 0.5 * (matrix.lam + fiber.stiffness_tensor()[2, 2, 2, 2])
mu0 = 0.5 * (matrix.mu + fiber.G_LT)
G_glob = build_green_operator(xi_flat, lam0, mu0, scheme="rotated", dx=dx)

l0 = 3.0 * dx[0]
Gc_mat = 0.8e-3
Gc_interphase = 0.4e-3
Gc_yarn = Gc_mat * 2.0
Gc_field = jnp.where(matrix_mask, Gc_mat, jnp.where(interphase_mask, Gc_interphase, Gc_yarn))

print("l0 [mm]:", l0)
print("Gc_field [MPa.mm] -- matrix:", Gc_mat, " interphase:", Gc_interphase, " fibre:", Gc_yarn)

## Staggered load-stepping loop

Uniaxial tension along x, ramped up over a handful of strain increments (see `configs/pff_prototype.yaml`
for the full adaptive-timestepper version up to `eps_goal=1e-2`; this notebook uses a fixed, coarser
ramp to keep it a quick, directly-runnable walkthrough). Each increment alternates a mechanical
solve (matrix+interphase stiffness degraded by the current damage) and a Helmholtz damage solve
with `update_history_hybrid` irreversibility, until the damage field stops changing.

In [ ]:
TOLER_LIN = 1e-2
TOLER_HELM = 1e-2
MAXITER_CG = 200
MAXITER_HELM = 200
MAXITER_STAGGER = 30
TOLER_STAGGER = 1e-2

d_field = jnp.zeros(Nv)
H_field = jnp.zeros(Nv)
eps_dir = jnp.array([[1.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
strain_levels = np.linspace(0, 5.0e-3, 20)

results = []
for k, eps_scale in enumerate(strain_levels):
    eps_bar = float(eps_scale) * eps_dir
    d_st = d_field
    for it_st in range(1, MAXITER_STAGGER + 1):
        d_prev = d_st

        g = jnp.where(damage_zone, degradation(d_st), 1.0)
        C_eff = g[None, None, None, None, :] * C_field

        eps, sigma, delta, conv_mech = solve_elastic(
            n, C_eff, G_glob, eps_bar, toler_lin=TOLER_LIN, maxiter=MAXITER_CG,
        )

        psi_pos, _ = strain_energy_amor_split(eps, lam_vox, mu_vox)
        psi_pos = jnp.where(damage_zone, psi_pos, 0.0)
        H_field = update_history_hybrid(H_field, psi_pos, d_prev)

        d_new, conv_helm = solve_helmholtz_cg_het(
            H_field, xi_flat, n, l0, Gc_field, d_prev, toler_cg=TOLER_HELM, maxiter=MAXITER_HELM,
        )
        d_st = d_new

        change = float(jnp.max(jnp.abs(d_st - d_prev)))
        if change < TOLER_STAGGER:
            break

    assert bool(conv_mech), f"mechanical CG did not converge at step {k+1}"
    d_field = d_st
    sigma11_ave = float(jnp.mean(sigma[0, 0]))
    results.append({"eps11": float(eps_scale), "sigma11_ave": sigma11_ave, "max_d": float(jnp.max(d_field))})
    print(f"step {k+1}/{len(strain_levels)}  eps11={float(eps_scale):.2e}  "
          f"sigma11_ave={sigma11_ave:8.3f} MPa  max(d)={float(jnp.max(d_field)):.4f}  "
          f"staggered_iters={it_st}")

print("\nPASSED -- mechanical CG converged at every step.")

## Visualize: macroscopic response and damage field

In [ ]:
eps11 = [r["eps11"] for r in results]
sigma11 = [r["sigma11_ave"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].plot([0] + eps11, [0] + sigma11, "o-")
axes[0].set_xlabel(r"$\bar\varepsilon_{11}$")
axes[0].set_ylabel(r"$\bar\sigma_{11}$ [MPa]")
axes[0].set_title("Macroscopic response")

d_grid = np.asarray(d_field).reshape(n)
im = axes[1].imshow(d_grid[:, :, 0].T, origin="lower", vmin=0, vmax=1, cmap="plasma",
                     extent=[0, n[0]*dx[0], 0, n[1]*dx[1]])
axes[1].set_title("Damage field d")
axes[1].set_xlabel("x [mm]")
axes[1].set_ylabel("y [mm]")
fig.colorbar(im, ax=axes[1], label="d")

fig.tight_layout()
plt.show()

## Post-processing: export to XDMF/HDF5

Same field-writing convention as the rest of the project (`post.io.IncrementalWriter`), so the
result can be opened in ParaView (`Xdmf3ReaderT`) alongside the other examples' output.

In [ ]:
eps_grid = field_to_grid(eps, n)
sigma_grid = field_to_grid(sigma, n)
vm_grid = von_mises(sigma_grid)

output_dir = "../output"
import os
os.makedirs(output_dir, exist_ok=True)

with IncrementalWriter(f"{output_dir}/pff_damage_patch", grid_shape=n, grid_length=L) as w:
    w.write_increment(0, {
        "phase":     phase_np.astype(np.float64).reshape(n),
        "strain":    to_voigt(eps_grid).astype(np.float64),
        "stress":    to_voigt(sigma_grid).astype(np.float64),
        "von_mises": vm_grid.astype(np.float64),
        "damage":    d_grid.astype(np.float64),
    }, time=0.0)

print(f"Wrote {output_dir}/pff_damage_patch.h5")
print(f"      {output_dir}/pff_damage_patch.xdmf")
print("Open the .xdmf in ParaView with the 'Xdmf3ReaderT' reader.")

## Next steps

- Try a different `seed`/`phi_target` in the RVE generation cell above -- fibre packing and volume
  fraction both shift damage nucleation sites and the macroscopic response, same as switching
  between real patches from `data/patches_fft/` would (see `configs/user/pff_prototype.yaml`).
- Compare against the idealized sphere-seed example in `examples/pff_damage.py` --
  real, sharp-interface microstructure typically shows more localized, geometry-driven
  damage nucleation than an idealized seed does.
- See `configs/user/pff_prototype.yaml` for the full production setup with an adaptive timestepper,
  driven via `scripts/simulation/pff_nw_cg_strain.py`.